# 12 — Segmentation Model Training (Stage 3)

Full model validation for the three-experiment design:

| Experiment | Input | Channels | Purpose |
|---|---|---|---|
| **A** | Single-date (Jul 30) | 9 | Conventional baseline |
| **B** | 4 phenological dates (Jan, Mar, Jul, Nov) | 36 | Multi-temporal naive |
| **C** | Stage 2 CNN forward-selection output | K* | Proposed method |

Each experiment × 2 architectures (DeepLabV3+CBAM, SegFormer) = **6 total runs**.

**Prerequisites**: run `02_image_processing.ipynb` and `04_feature_analysis.ipynb` first.

---

In [ ]:
import sys, os, re, time
from glob import glob
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
import rasterio
import mlflow

sys.path.append("../")

from geoai.geoai.train import RasterPatchDataset, train_semantic_one_epoch
from geoai.geoai.utils.device import get_device
from src.models import DeepLabV3PlusCBAM, build_segformer

DEVICE = get_device()
print(f"Device: {DEVICE}")
print(f"PyTorch: {torch.__version__}")

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
PROJECT_ROOT     = os.path.abspath("../")
PROCESSED_DIR    = os.path.join(PROJECT_ROOT, "data/processed")
S2_PROCESSED_DIR = os.path.join(PROCESSED_DIR, "s2")
CDL_FILTERED     = os.path.join(PROCESSED_DIR, "cdl/cdl_2022_study_area_filtered.tif")
MODELS_DIR       = os.path.join(PROJECT_ROOT, "models")
STAGE2_CSV       = os.path.join(PROCESSED_DIR, "stage2_history.csv")

os.makedirs(MODELS_DIR, exist_ok=True)

# ── S2 band metadata ──────────────────────────────────────────────────────────
# GEE export order: B1, B2 … B7, B8, B8A, B11, B12  (11 bands)
S2_BAND_NAMES    = ["B1", "B2", "B3", "B4", "B5", "B6", "B7", "B8", "B8A", "B11", "B12"]
N_BANDS_PER_DATE = len(S2_BAND_NAMES)

# 9 vegetation bands used for Exp A and B (excludes coastal B1 and redundant B8A)
VEGE_BANDS = ["B2", "B3", "B4", "B5", "B6", "B7", "B8", "B11", "B12"]

# ── CDL classes ───────────────────────────────────────────────────────────────
KEEP_CLASSES = [3, 6, 24, 36, 37, 54, 61, 69, 75, 76, 220]
CLASS_NAMES  = {
    3:   "Rice",        6:   "Sunflower",    24:  "Winter Wheat",
    36:  "Alfalfa",     37:  "Other Hay",     54:  "Tomatoes",
    61:  "Fallow/Idle", 69:  "Grapes",        75:  "Almonds",
    76:  "Walnuts",     220: "Plums",
}
CLASS_REMAP  = {cls_id: i + 1 for i, cls_id in enumerate(KEEP_CLASSES)}
NUM_CLASSES  = len(KEEP_CLASSES) + 1   # 12 (0 = background)

REMAP_LUT = np.zeros(256, dtype=np.int64)
for cdl_id, model_id in CLASS_REMAP.items():
    if cdl_id < 256:
        REMAP_LUT[cdl_id] = model_id

# ── MLflow ────────────────────────────────────────────────────────────────────
MLFLOW_TRACKING_URI = "file://" + os.path.join(PROJECT_ROOT, "mlruns")
MLFLOW_EXPERIMENT   = "research-crop-mapping"
# For remote server: MLFLOW_TRACKING_URI = "http://mlflow-geoai.stelarea.com"

# ── Training hyperparameters ─────────────────────────────────────────────────
PATCH_SIZE     = 128     # 256 for full run; 128 for fast iteration
STRIDE         = 128     # non-overlapping
MIN_VALID_FRAC = 0.3
BATCH_SIZE     = 8
MAX_EPOCHS     = 15      # 100 for full run; 15 for small-scale test
EARLY_STOP     = 5       # 10 for full run; 5 for small-scale test
VAL_FRAC       = 0.15
TEST_FRAC      = 0.15
SEED           = 42

SMALL_SCALE = True   # Set False for full production run

# Per-architecture hyperparameters (from methodology.tex)
ARCH_CFG = {
    "deeplabv3plus_cbam": {"lr": 1e-4,  "weight_decay": 1e-4,  "encoder": "resnet50"},
    "segformer":          {"lr": 6e-5,  "weight_decay": 1e-2,  "encoder": "mit_b2"},
}

# ── Verify paths ─────────────────────────────────────────────────────────────
S2_PROCESSED = sorted(glob(f"{S2_PROCESSED_DIR}/*_processed.tif"))
print(f"S2 processed files : {len(S2_PROCESSED)}")
print(f"CDL filtered       : {os.path.exists(CDL_FILTERED)}")
for p in S2_PROCESSED:
    print(f"  {os.path.basename(p)}")
print(f"\nConfig: PATCH_SIZE={PATCH_SIZE}, MAX_EPOCHS={MAX_EPOCHS}, EARLY_STOP={EARLY_STOP}")
print(f"SMALL_SCALE={SMALL_SCALE}  (set to False for production run)")


In [ ]:
# ── Load Stage 2 results ──────────────────────────────────────────────────────
stage2_bands   = None
stage2_run_id  = None

if os.path.exists(STAGE2_CSV):
    hist_df      = pd.read_csv(STAGE2_CSV)
    stage2_bands = hist_df[hist_df["accepted"] == True]["band"].tolist()
    print(f"Stage 2 — {len(stage2_bands)} bands selected:")
    for b in stage2_bands:
        print(f"  {b}")
else:
    print(f"⚠️  {STAGE2_CSV} not found. Experiment C will be skipped.")

# ── Build global band name → stacked-array index map ─────────────────────────
def parse_date(path):
    m = re.search(r"_(\d{4})_(\d{2})_(\d{2})_processed", os.path.basename(path))
    return f"{m.group(1)}{m.group(2)}{m.group(3)}" if m else None

all_band_names   = []
date_to_file_idx = {}
for i, path in enumerate(S2_PROCESSED):
    d = parse_date(path)
    date_to_file_idx[d] = i
    all_band_names.extend([f"{b}_{d}" for b in S2_BAND_NAMES])

band_name_to_idx = {name: i for i, name in enumerate(all_band_names)}
total_channels   = len(all_band_names)
available_dates  = sorted(date_to_file_idx.keys())

print(f"\nTotal stacked channels: {total_channels} ({len(S2_PROCESSED)} dates × {N_BANDS_PER_DATE} bands)")
print(f"Available dates: {available_dates}")

# ── Exp A: best available single date ────────────────────────────────────────
# Prefer Jul 30; fall back to last available date (most likely to have crops)
july30_key = next(
    (k for k in available_dates if k[4:6] == "07" and k[6:8] in ("29", "30")),
    None,
)
if july30_key is None:
    july30_key = available_dates[-1]   # last available date as fallback
    print(f"⚠️  Jul 30 not available — using fallback date: {july30_key}")

file_off_A  = date_to_file_idx[july30_key] * N_BANDS_PER_DATE
exp_A_idx   = [file_off_A + S2_BAND_NAMES.index(b) for b in VEGE_BANDS]
exp_A_names = [f"{b}_{july30_key}" for b in VEGE_BANDS]
print(f"\nExp A — date: {july30_key}, channels: {len(exp_A_idx)}")

# ── Exp B: phenological dates × 9 vege bands ─────────────────────────────────
# Prefer Jan, Mar, Jul, Nov; use closest available date for each target month
phenol_targets = {"Jan": [1], "Mar": [3], "Jul": [7, 8], "Nov": [11, 10]}
phenol_map = {}
for label, months in phenol_targets.items():
    match = next(
        (d for d in available_dates if int(d[4:6]) in months),
        None,
    )
    if match:
        phenol_map[label] = match

if len(phenol_map) < 2:
    # Not enough distinct phenological dates — use all available dates
    phenol_map = {f"D{i}": d for i, d in enumerate(available_dates)}
    print(f"⚠️  Fewer than 2 phenological months found — using all {len(available_dates)} dates")

exp_B_idx, exp_B_names = [], []
for target, d in phenol_map.items():
    off = date_to_file_idx[d] * N_BANDS_PER_DATE
    exp_B_idx   += [off + S2_BAND_NAMES.index(b) for b in VEGE_BANDS]
    exp_B_names += [f"{b}_{d}" for b in VEGE_BANDS]

# De-duplicate (same date may appear for multiple targets when dates are limited)
seen, exp_B_idx_dedup, exp_B_names_dedup = set(), [], []
for idx, name in zip(exp_B_idx, exp_B_names):
    if idx not in seen:
        seen.add(idx)
        exp_B_idx_dedup.append(idx)
        exp_B_names_dedup.append(name)
exp_B_idx, exp_B_names = exp_B_idx_dedup, exp_B_names_dedup

print(f"Exp B — dates: {phenol_map}, channels: {len(exp_B_idx)}")

# ── Exp C: Stage 2 selected bands ────────────────────────────────────────────
if stage2_bands:
    exp_C_idx   = [band_name_to_idx[b] for b in stage2_bands if b in band_name_to_idx]
    exp_C_names = [b for b in stage2_bands if b in band_name_to_idx]
    if not exp_C_idx:
        print("⚠️  No Stage 2 bands match current processed files — Exp C skipped.")
        exp_C_idx = exp_C_names = None
    else:
        print(f"Exp C — channels: {len(exp_C_idx)}")
else:
    exp_C_idx   = None
    exp_C_names = None


In [ ]:
# ── Compute inverse-frequency class weights from CDL raster ──────────────────
with rasterio.open(CDL_FILTERED) as src:
    cdl_arr = src.read(1).astype(np.int32)

class_counts = np.zeros(NUM_CLASSES, dtype=np.float64)
class_counts[0] = (cdl_arr == 0).sum()   # background
for cdl_id, model_id in CLASS_REMAP.items():
    class_counts[model_id] += (cdl_arr == cdl_id).sum()

freq          = class_counts / (class_counts.sum() + 1e-9)
class_weights = 1.0 / (freq + 1e-9)
class_weights /= class_weights.sum()   # normalise so weights sum to 1
CLASS_WEIGHTS_TENSOR = torch.tensor(class_weights, dtype=torch.float32)

print("Class weights (inverse-frequency, normalised):")
print(f"  [0] Background: count={class_counts[0]:10.0f}  weight={class_weights[0]:.5f}")
for i, cdl_id in enumerate(KEEP_CLASSES, start=1):
    name = CLASS_NAMES.get(cdl_id, f"Class {cdl_id}")
    print(f"  [{i:2d}] {name:20s}: count={class_counts[i]:10.0f}  weight={class_weights[i]:.5f}")

In [ ]:
# ── Training helpers ──────────────────────────────────────────────────────────

def compute_miou(logits, labels, num_classes, ignore_index=0):
    """mIoU over foreground classes (class 0 skipped)."""
    preds  = logits.argmax(dim=1).view(-1).cpu().numpy()
    labels = labels.view(-1).cpu().numpy()
    ious = []
    for cls in range(1, num_classes):
        p = (preds == cls)
        l = (labels == cls)
        inter = (p & l).sum()
        union = (p | l).sum()
        if union > 0:
            ious.append(inter / union)
    return float(np.mean(ious)) if ious else 0.0


def compute_per_class_iou(logits, labels, num_classes):
    """Per-class IoU dict {model_class_id: iou}. Background (0) excluded."""
    preds  = logits.argmax(dim=1).view(-1).numpy()
    labels = labels.view(-1).numpy()
    ious = {}
    for cls in range(1, num_classes):
        p = (preds == cls)
        l = (labels == cls)
        inter = (p & l).sum()
        union = (p | l).sum()
        ious[cls] = float(inter / union) if union > 0 else float("nan")
    return ious


@torch.no_grad()
def validate_one_epoch(model, loader, criterion, device, num_classes):
    """Run validation, return dict with loss, miou, oa."""
    model.eval()
    total_loss   = 0.0
    all_logits   = []
    all_labels   = []

    for imgs, masks in loader:
        imgs, masks = imgs.to(device), masks.to(device)
        logits       = model(imgs)
        total_loss  += criterion(logits, masks).item()
        all_logits.append(logits.cpu())
        all_labels.append(masks.cpu())

    all_logits = torch.cat(all_logits, dim=0)
    all_labels = torch.cat(all_labels, dim=0)

    preds = all_logits.argmax(dim=1)
    oa    = (preds == all_labels).float().mean().item()
    miou  = compute_miou(all_logits, all_labels, num_classes)

    return {
        "loss": total_loss / len(loader),
        "miou": miou,
        "oa":   oa,
    }


@torch.no_grad()
def evaluate_test_set(model, loader, num_classes):
    """Full test-set evaluation: per-class IoU, mIoU, OA."""
    model.eval()
    all_logits = []
    all_labels = []

    for imgs, masks in loader:
        logits = model(imgs.to(DEVICE))
        all_logits.append(logits.cpu())
        all_labels.append(masks.cpu())

    all_logits = torch.cat(all_logits, dim=0)
    all_labels = torch.cat(all_labels, dim=0)

    return {
        "miou":          compute_miou(all_logits, all_labels, num_classes),
        "oa":            (all_logits.argmax(1) == all_labels).float().mean().item(),
        "per_class_iou": compute_per_class_iou(all_logits, all_labels, num_classes),
    }

In [ ]:
# ── Model builder ─────────────────────────────────────────────────────────────

def build_model(arch, in_channels, num_classes):
    cfg = ARCH_CFG[arch]
    if arch == "deeplabv3plus_cbam":
        model = DeepLabV3PlusCBAM(
            encoder_name=cfg["encoder"],
            encoder_weights="imagenet",
            in_channels=in_channels,
            num_classes=num_classes,
        )
    elif arch == "segformer":
        model = build_segformer(
            encoder_name=cfg["encoder"],
            encoder_weights="imagenet",
            in_channels=in_channels,
            num_classes=num_classes,
        )
    else:
        raise ValueError(f"Unknown architecture: {arch}")

    n_params = sum(p.numel() for p in model.parameters())
    print(f"  {arch} ({cfg['encoder']}): {n_params:,} parameters")
    return model.to(DEVICE)


# ── Main training runner ──────────────────────────────────────────────────────

def run_experiment(
    exp_name,
    arch,
    band_indices,
    band_names_list,
    description,
    stage2_run_id=None,
):
    """
    Train one experiment configuration and log everything to MLflow.

    Returns dict with experiment summary metrics.
    """
    cfg         = ARCH_CFG[arch]
    in_channels = len(band_indices)
    exp_dir     = os.path.join(MODELS_DIR, exp_name)
    best_ckpt   = os.path.join(exp_dir, "best_model.pth")
    os.makedirs(exp_dir, exist_ok=True)

    print(f"\n{'='*65}")
    print(f" {exp_name}")
    print(f"  arch={arch}  in_channels={in_channels}")
    print(f"  {description}")
    print(f"{'='*65}\n")

    # ── Dataset ───────────────────────────────────────────────────────────────
    dataset = RasterPatchDataset(
        s2_paths=S2_PROCESSED,
        cdl_path=CDL_FILTERED,
        patch_size=PATCH_SIZE,
        stride=STRIDE,
        keep_classes=KEEP_CLASSES,
        remap_lut=REMAP_LUT,
        min_valid_frac=MIN_VALID_FRAC,
        band_indices=band_indices,
    )

    n_total = len(dataset)
    n_test  = max(1, int(TEST_FRAC  * n_total))
    n_val   = max(1, int(VAL_FRAC   * n_total))
    n_train = n_total - n_val - n_test
    assert n_train > 0, f"Not enough patches: {n_total}"

    gen = torch.Generator().manual_seed(SEED)
    train_ds, val_ds, test_ds = random_split(
        dataset, [n_train, n_val, n_test], generator=gen
    )

    # num_workers=0: rasterio handles cannot be pickled for forked workers
    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
    val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    test_dl  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    print(f"  Patches: {n_train} train / {n_val} val / {n_test} test")

    # ── Model + optimiser + scheduler + loss ──────────────────────────────────
    model     = build_model(arch, in_channels, NUM_CLASSES)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"]
    )
    scheduler = torch.optim.lr_scheduler.PolynomialLR(
        optimizer, total_iters=MAX_EPOCHS, power=0.9
    )
    criterion = nn.CrossEntropyLoss(
        weight=CLASS_WEIGHTS_TENSOR.to(DEVICE)
    )

    # ── MLflow run ────────────────────────────────────────────────────────────
    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
    mlflow.set_experiment(MLFLOW_EXPERIMENT)
    timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")

    with mlflow.start_run(run_name=f"{exp_name}_{timestamp}") as run:
        mlflow.log_params({
            "experiment":     exp_name,
            "architecture":   arch,
            "encoder":        cfg["encoder"],
            "in_channels":    in_channels,
            "num_classes":    NUM_CLASSES,
            "patch_size":     PATCH_SIZE,
            "batch_size":     BATCH_SIZE,
            "max_epochs":     MAX_EPOCHS,
            "early_stopping": EARLY_STOP,
            "learning_rate":  cfg["lr"],
            "weight_decay":   cfg["weight_decay"],
            "optimizer":      "AdamW",
            "lr_scheduler":   "PolynomialLR(power=0.9)",
            "loss":           "WeightedCrossEntropy",
            "train_patches":  n_train,
            "val_patches":    n_val,
            "test_patches":   n_test,
            "description":    description,
            "keep_classes":   str(KEEP_CLASSES),
        })
        mlflow.set_tag("band_names", str(band_names_list))
        mlflow.set_tag("n_bands",    str(in_channels))
        if stage2_run_id:
            mlflow.set_tag("stage2_run_id", stage2_run_id)

        # ── Training loop ─────────────────────────────────────────────────────
        best_miou  = 0.0
        no_improve = 0
        history    = []
        t_start    = time.time()

        for epoch in range(MAX_EPOCHS):
            t_ep = time.time()

            train_loss = train_semantic_one_epoch(
                model, optimizer, train_dl, DEVICE, epoch, criterion,
                print_freq=len(train_dl),   # print once per epoch
                verbose=False,
            )
            val_m   = validate_one_epoch(model, val_dl, criterion, DEVICE, NUM_CLASSES)
            scheduler.step()

            ep_t = time.time() - t_ep
            mlflow.log_metrics({
                "train_loss": train_loss,
                "val_loss":   val_m["loss"],
                "val_miou":   val_m["miou"],
                "val_oa":     val_m["oa"],
                "lr":         scheduler.get_last_lr()[0],
            }, step=epoch)

            history.append({
                "epoch":      epoch + 1,
                "train_loss": round(train_loss,       4),
                "val_loss":   round(val_m["loss"],    4),
                "val_miou":   round(val_m["miou"],    4),
                "val_oa":     round(val_m["oa"],      4),
                "epoch_t_s":  round(ep_t,              1),
            })

            if val_m["miou"] > best_miou:
                best_miou  = val_m["miou"]
                no_improve = 0
                torch.save({
                    "epoch":            epoch,
                    "model_state_dict": model.state_dict(),
                    "best_miou":        best_miou,
                    "band_indices":     band_indices,
                    "band_names":       band_names_list,
                    "in_channels":      in_channels,
                    "num_classes":      NUM_CLASSES,
                    "architecture":     arch,
                }, best_ckpt)
            else:
                no_improve += 1

            total_min = (time.time() - t_start) / 60
            print(
                f"  Ep {epoch+1:3d}/{MAX_EPOCHS} "
                f"loss={train_loss:.4f} val_loss={val_m['loss']:.4f} "
                f"mIoU={val_m['miou']:.4f} best={best_miou:.4f} "
                f"patience={no_improve}/{EARLY_STOP} "
                f"{ep_t:.0f}s  total={total_min:.1f}min"
            )

            if no_improve >= EARLY_STOP:
                print(f"  Early stopping at epoch {epoch + 1}")
                break

        # ── Test evaluation ───────────────────────────────────────────────────
        ckpt = torch.load(best_ckpt, map_location=DEVICE)
        model.load_state_dict(ckpt["model_state_dict"])
        test_r = evaluate_test_set(model, test_dl, NUM_CLASSES)

        mlflow.log_metrics({
            "best_val_miou": best_miou,
            "test_miou":     test_r["miou"],
            "test_oa":       test_r["oa"],
            "total_epochs":  len(history),
        })

        # Per-class IoU
        for cls_id, iou in test_r["per_class_iou"].items():
            if not np.isnan(iou):
                name = CLASS_NAMES.get(KEEP_CLASSES[cls_id - 1], f"cls{cls_id}")
                mlflow.log_metric(
                    f"test_iou_{name.lower().replace('/', '_').replace(' ', '_')}",
                    iou,
                )

        # ── Artifacts ─────────────────────────────────────────────────────────
        hist_df  = pd.DataFrame(history)
        hist_csv = os.path.join(exp_dir, "training_history.csv")
        hist_df.to_csv(hist_csv, index=False)

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
        ax1.plot(hist_df["epoch"], hist_df["train_loss"], label="Train")
        ax1.plot(hist_df["epoch"], hist_df["val_loss"],   label="Val")
        ax1.set(xlabel="Epoch", ylabel="Loss", title=f"{exp_name} — Loss")
        ax1.legend(); ax1.grid(True)

        ax2.plot(hist_df["epoch"], hist_df["val_miou"], color="green", label="Val mIoU")
        ax2.axhline(best_miou, linestyle="--", color="gray", label=f"Best={best_miou:.4f}")
        ax2.set(xlabel="Epoch", ylabel="mIoU", title=f"{exp_name} — mIoU")
        ax2.legend(); ax2.grid(True)

        plt.tight_layout()
        curve_path = os.path.join(exp_dir, "training_curve.png")
        plt.savefig(curve_path, dpi=150)
        plt.show()

        mlflow.log_artifact(best_ckpt)
        mlflow.log_artifact(hist_csv)
        mlflow.log_artifact(curve_path)

        run_id = run.info.run_id

    print(f"\n✅ {exp_name}  val_mIoU={best_miou:.4f}  test_mIoU={test_r['miou']:.4f}  run={run_id}")

    return {
        "exp_name":      exp_name,
        "arch":          arch,
        "in_channels":   in_channels,
        "best_val_miou": round(best_miou,      4),
        "test_miou":     round(test_r["miou"], 4),
        "test_oa":       round(test_r["oa"],   4),
        "total_epochs":  len(history),
        "run_id":        run_id,
        "ckpt":          best_ckpt,
    }

## Experiment A — Single-Date Baseline

9 vegetation bands from **Jul 30** (peak season, 99.6% valid coverage).  
Represents the conventional single-date approach without temporal information.

In [ ]:
results_A_dlv3 = run_experiment(
    exp_name     = "exp_A_deeplabv3plus_cbam",
    arch         = "deeplabv3plus_cbam",
    band_indices = exp_A_idx,
    band_names_list = exp_A_names,
    description  = f"Single-date {july30_key}, 9ch, DeepLabV3+CBAM",
)

In [ ]:
results_A_sfm = run_experiment(
    exp_name     = "exp_A_segformer",
    arch         = "segformer",
    band_indices = exp_A_idx,
    band_names_list = exp_A_names,
    description  = f"Single-date {july30_key}, 9ch, SegFormer",
)

## Experiment B — Multi-Temporal Naive Baseline

9 vegetation bands × 4 phenological dates = **36 channels**.  
Dates chosen by calendar (Jan, Mar, Jul, Nov) — no selection algorithm.

In [ ]:
results_B_dlv3 = run_experiment(
    exp_name     = "exp_B_deeplabv3plus_cbam",
    arch         = "deeplabv3plus_cbam",
    band_indices = exp_B_idx,
    band_names_list = exp_B_names,
    description  = f"4 phenological dates {list(phenol_map.values())}, 36ch, DeepLabV3+CBAM",
)

In [ ]:
results_B_sfm = run_experiment(
    exp_name     = "exp_B_segformer",
    arch         = "segformer",
    band_indices = exp_B_idx,
    band_names_list = exp_B_names,
    description  = f"4 phenological dates {list(phenol_map.values())}, 36ch, SegFormer",
)

## Experiment C — Multi-Temporal with Band Selection (Proposed Method)

**K\*** bands from Stage 2 CNN forward selection.  
Stage 1 determined order; Stage 2 determined the optimal cut-off.

In [ ]:
if exp_C_idx is None:
    print("⚠️  Experiment C skipped — run 04_feature_analysis.ipynb to generate Stage 2 results.")
else:
    results_C_dlv3 = run_experiment(
        exp_name        = "exp_C_deeplabv3plus_cbam",
        arch            = "deeplabv3plus_cbam",
        band_indices    = exp_C_idx,
        band_names_list = exp_C_names,
        description     = f"Stage2 selection K*={len(exp_C_idx)}ch, DeepLabV3+CBAM",
        stage2_run_id   = stage2_run_id,
    )

In [ ]:
if exp_C_idx is None:
    print("⚠️  Experiment C skipped.")
else:
    results_C_sfm = run_experiment(
        exp_name        = "exp_C_segformer",
        arch            = "segformer",
        band_indices    = exp_C_idx,
        band_names_list = exp_C_names,
        description     = f"Stage2 selection K*={len(exp_C_idx)}ch, SegFormer",
        stage2_run_id   = stage2_run_id,
    )

## Results Summary

In [ ]:
# Collect all results that were actually run
all_results = []
for var_name in [
    "results_A_dlv3", "results_A_sfm",
    "results_B_dlv3", "results_B_sfm",
    "results_C_dlv3", "results_C_sfm",
]:
    r = globals().get(var_name)
    if r is not None:
        all_results.append(r)

summary_df = pd.DataFrame(all_results)[[
    "exp_name", "arch", "in_channels",
    "best_val_miou", "test_miou", "test_oa", "total_epochs", "run_id"
]]
summary_df = summary_df.sort_values("test_miou", ascending=False)

print("\n=== Experiment Results Summary ===")
print(summary_df.to_string(index=False))

summary_csv = os.path.join(MODELS_DIR, "experiment_summary.csv")
summary_df.to_csv(summary_csv, index=False)
print(f"\nSaved: {summary_csv}")

In [ ]:
# Comparison bar chart: test mIoU per experiment × architecture
if len(summary_df) > 0:
    fig, ax = plt.subplots(figsize=(10, 5))

    x      = np.arange(len(summary_df))
    labels = [
        f"{r['exp_name'].replace('exp_', '').replace('_deeplabv3plus_cbam', '\nDL').replace('_segformer', '\nSF')}"
        for _, r in summary_df.iterrows()
    ]
    colors = [
        "steelblue" if "deeplabv3" in r["arch"] else "darkorange"
        for _, r in summary_df.iterrows()
    ]

    bars = ax.bar(x, summary_df["test_miou"], color=colors, edgecolor="white", width=0.6)
    for bar, val in zip(bars, summary_df["test_miou"]):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.002,
                f"{val:.4f}", ha="center", va="bottom", fontsize=9)

    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=9)
    ax.set_ylabel("Test mIoU")
    ax.set_title("Experiment Comparison — Test mIoU (DeepLabV3+CBAM vs SegFormer)")
    ax.set_ylim(0, min(1.0, summary_df["test_miou"].max() + 0.1))
    ax.grid(axis="y", alpha=0.4)

    from matplotlib.patches import Patch
    ax.legend(handles=[
        Patch(color="steelblue",  label="DeepLabV3+CBAM"),
        Patch(color="darkorange", label="SegFormer"),
    ])

    plt.tight_layout()
    chart_path = os.path.join(MODELS_DIR, "experiment_comparison.png")
    plt.savefig(chart_path, dpi=150)
    plt.show()
    print(f"Saved: {chart_path}")